# TEST-02: Always-Flat Signal

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Verify that a constant 0 signal produces zero activity.

**Expected:**
- Final equity = 10,000 exactly
- Trade count = 0, gross PnL = 0, fees = 0, funding = 0
- Max drawdown = 0
- Sharpe = 0.0 (sigma=0 path)
- Cost drag = NaN (gross PnL = 0)

In [1]:
import sys, math
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

from lib.backtest import run_backtest
from lib.evaluation import max_drawdown, sharpe, cost_drag, _returns_from_equity

OHLCV_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet'
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10_000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

ohlcv = load_ohlcv(OHLCV_PATH)
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))
sig = pd.Series(0.0, index=ohlcv.index, name='signal')
result = run_backtest(sig, ohlcv, funding, **COSTS, execution='next_bar_open',
                      stream='klines', interval='4h', gaps_path=GAPS_PATH)

In [2]:
ret = _returns_from_equity(result.equity_curve)
sr = sharpe(ret, '4h')
dd, _, _ = max_drawdown(result.equity_curve)
cd = cost_drag(result)

print(f'Final equity:     {result.equity_curve.iloc[-1]:.6f} (expected: 10000.000000)')
print(f'Trade count:      {len(result.trades_df)} (expected: 0)')
print(f'Realized PnL:     {result.realized_pnl:.6f} (expected: 0)')
print(f'Unrealized PnL:   {result.unrealized_pnl:.6f} (expected: 0)')
print(f'Fees paid:        {result.fees_paid:.6f} (expected: 0)')
print(f'Funding paid:     {result.funding_paid:.6f} (expected: 0)')
print(f'Max drawdown:     {dd*100:.4f}% (expected: 0%)')
print(f'Sharpe:           {sr:.4f} (expected: 0.0 -- sigma=0 branch)')
print(f'Cost drag:        {cd["cost_drag_fraction"]} (expected: nan -- gross_pnl=0)')

Final equity:     10000.000000 (expected: 10000.000000)
Trade count:      0 (expected: 0)
Realized PnL:     0.000000 (expected: 0)
Unrealized PnL:   0.000000 (expected: 0)
Fees paid:        0.000000 (expected: 0)
Funding paid:     0.000000 (expected: 0)
Max drawdown:     0.0000% (expected: 0%)
Sharpe:           0.0000 (expected: 0.0 -- sigma=0 branch)
Cost drag:        nan (expected: nan -- gross_pnl=0)


In [3]:
equity_ok = abs(result.equity_curve.iloc[-1] - 10_000.0) < 0.01
trades_ok = len(result.trades_df) == 0
all_zero_ok = (result.realized_pnl == 0.0 and result.unrealized_pnl == 0.0
               and result.fees_paid == 0.0 and result.funding_paid == 0.0)
dd_ok = dd == 0.0
verdict = 'PASS' if (equity_ok and trades_ok and all_zero_ok and dd_ok) else 'FAIL'

print(f'=== VERDICT: {verdict} ===')
print(f'Equity = 10000:   {"PASS" if equity_ok else "FAIL"}')
print(f'Trade count = 0:  {"PASS" if trades_ok else "FAIL"}')
print(f'All costs zero:   {"PASS" if all_zero_ok else "FAIL"}')
print(f'Max DD = 0:       {"PASS" if dd_ok else "FAIL"}')
print(f'Sharpe = 0.0:     {"PASS" if sr == 0.0 else "FAIL"} (sigma=0 returns 0.0, not NaN)')
print(f'Cost drag = nan:  {"PASS" if not math.isfinite(cd["cost_drag_fraction"]) else "FAIL"}')
print()
print('Edge-case note: When gross_pnl=0, check_promotion_gates casts nan to 0.0')
print('                so Gate 5 (cost_drag <= 0.50) PASSES for an always-flat strategy.')
print('                This is a known edge case (documented in TEST-07).')

=== VERDICT: PASS ===
Equity = 10000:   PASS
Trade count = 0:  PASS
All costs zero:   PASS
Max DD = 0:       PASS
Sharpe = 0.0:     PASS (sigma=0 returns 0.0, not NaN)
Cost drag = nan:  PASS

Edge-case note: When gross_pnl=0, check_promotion_gates casts nan to 0.0
                so Gate 5 (cost_drag <= 0.50) PASSES for an always-flat strategy.
                This is a known edge case (documented in TEST-07).
